#### Converting Original Dataset into NumpyArray

In [1]:
import pandas as pd
import numpy as np
import os

# Step 1: Load metadata
def load_metadata(csv_path):
    """
    Reads metadata.csv with columns: filename,label
    Returns DataFrame.
    """
    df = pd.read_csv(csv_path)
    return df

# Step 2: Compute mean & std 
def compute_mean_std(df, root_dir="."):
    """
    Compute global mean and std over training set only.
    df: metadata DataFrame (assumed already split; for now you can pass full df).
    root_dir: root directory where files are located.
    """
    sums = 0.0
    squared_sums = 0.0
    count = 0
    
    for path in df['filename']:
        file_path = os.path.join(root_dir, path)
        mel = np.load(file_path).astype(np.float32)
        
        sums += mel.sum()
        squared_sums += (mel ** 2).sum()
        count += mel.size
    
    mean = sums / count
    std = np.sqrt((squared_sums / count) - (mean ** 2))
    
    return mean, std

# ========= Step 3: Loader & Preprocessor =========
def load_and_preprocess(df, mean=None, std=None, root_dir="."):
    """
    Loads .npy files, reshapes to (H,W,1), normalizes.
    If mean/std provided → global normalization
    Else → per-sample normalization
    """
    X, y = [], []
    
    for _, row in df.iterrows():
        file_path = os.path.join(root_dir, row['filename'])
        mel = np.load(file_path).astype(np.float32)
        
        # Reshape to (H,W,1)
        mel = mel[..., np.newaxis]
        
        # Normalization
        if mean is not None and std is not None:
            mel = (mel - mean) / std   # global normalization
        else:
            mel = mel / (np.max(mel) + 1e-8)  # per-sample scaling
        
        X.append(mel)
        y.append(row['label'])
    
    X = np.array(X)
    y = np.array(y)
    return X, y

# ========= Example Usage =========
csv_path = "final_metadata_balanced_fixed.csv"   # <- after you fixed paths
df = load_metadata(csv_path)

# First: compute mean/std (on train split ideally)
mean, std = compute_mean_std(df)
print("Global mean:", mean, "Global std:", std)

# Save mean/std for later use (important for deployment)
np.savez("normalization_stats.npz", mean=mean, std=std)

# Then: load full dataset with global normalization
X, y = load_and_preprocess(df, mean, std)

print("X shape:", X.shape)   # (N, H, W, 1)
print("y shape:", y.shape)

Global mean: -49.602452774109494 Global std: 20.54078017353221
X shape: (1580, 128, 94, 1)
y shape: (1580,)


#### Spliiting Dataset into Training,Testing and Validation

In [2]:
from sklearn.model_selection import train_test_split
import pandas as pd

# Load metadata (already fixed paths)
df = pd.read_csv("final_metadata_balanced_fixed.csv")

# Step 1: Train/Test split (85% train_val, 15% test)
train_val_df, test_df = train_test_split(
    df,
    test_size=0.15,
    stratify=df['label'],
    random_state=42
)

# Step 2: Split train into Train/Val (train=70%, val=15%)
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.1765,   # 0.1765 * 0.85 ≈ 0.15 of total
    stratify=train_val_df['label'],
    random_state=42
)

# Check distribution
print("Train samples:", len(train_df), "Distress:", sum(train_df['label']==1), "Non-distress:", sum(train_df['label']==0))
print("Val samples:", len(val_df), "Distress:", sum(val_df['label']==1), "Non-distress:", sum(val_df['label']==0))
print("Test samples:", len(test_df), "Distress:", sum(test_df['label']==1), "Non-distress:", sum(test_df['label']==0))

# Save splits
train_df.to_csv("train1.csv", index=False)
val_df.to_csv("val1.csv", index=False)
test_df.to_csv("test1.csv", index=False)

print("✅ Saved train.csv, val.csv, test.csv")

Train samples: 1105 Distress: 553 Non-distress: 552
Val samples: 238 Distress: 119 Non-distress: 119
Test samples: 237 Distress: 118 Non-distress: 119
✅ Saved train.csv, val.csv, test.csv


#### Adding Weights to Handle the Class Imbalance

In [3]:
import pandas as pd
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Load your train split
train_df = pd.read_csv("train1.csv")

# Extract labels
y_train = train_df['label'].values

# Compute class weights
classes = np.unique(y_train)   # [0, 1]
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

# Convert to dictionary {class_label: weight}
class_weight_dict1 = dict(zip(classes, class_weights))

print("✅ Computed class weights:", class_weight_dict1)

✅ Computed class weights: {0: 1.0009057971014492, 1: 0.9990958408679927}


In [4]:
import numpy as np
import pandas as pd
import os
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# Load normalization stats
stats = np.load("normalization_stats.npz")
mean, std = stats["mean"], stats["std"]

# Load data splits
train_df = pd.read_csv("train1.csv")
val_df = pd.read_csv("val1.csv")
test_df = pd.read_csv("test1.csv")

# Data loader
def load_data(df, mean, std, root_dir="."):
    X, y = [], []
    for _, row in df.iterrows():
        file_path = os.path.join(root_dir, row['filename'])
        mel = np.load(file_path).astype(np.float32)
        mel = mel[..., np.newaxis]
        mel = (mel - mean) / std
        X.append(mel)
        y.append(row['label'])
    return np.array(X), np.array(y)

# Load train, val, test data
print("Loading data...")
X_train, y_train = load_data(train_df, mean, std)
X_val, y_val = load_data(val_df, mean, std)
X_test, y_test = load_data(test_df, mean, std)

print(f"Train shape: {X_train.shape}")
print(f"Val shape: {X_val.shape}")
print(f"Test shape: {X_test.shape}")

input_shape = X_train.shape[1:]  # (128, 94, 1)

# Compute class weights
from sklearn.utils.class_weight import compute_class_weight
classes = np.unique(y_train)
class_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))
print(f"Class weights: {class_weight_dict}")

# ========== Model : NO Attention-based CNN ==========
from tensorflow.keras import layers, models
import tensorflow as tf

def build_cnn_lstm(input_shape):
    inputs = layers.Input(shape=input_shape)

    # CNN blocks
    x = layers.Conv2D(64, (3,3), padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)

    x = layers.Conv2D(128, (3,3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)

    # reshape for LSTM
    shape = x.shape
    x = layers.Reshape((shape[1] * shape[2], shape[3]))(x)

    # LSTM
    x = layers.Bidirectional(layers.LSTM(128))(x)

    # Dense layers
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.4)(x)

    output = layers.Dense(1, activation="sigmoid")(x)

    return models.Model(inputs, output)


# ========== Training Function ==========
def train_and_evaluate(model, model_name):
    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print(f"{'='*60}")
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy', 
                 tf.keras.metrics.Precision(name='precision'),
                 tf.keras.metrics.Recall(name='recall'),
                 tf.keras.metrics.AUC(name='auc')]
    )
    
    callbacks = [
        EarlyStopping(monitor='val_auc', patience=10, restore_best_weights=True, mode='max', verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1),
        ModelCheckpoint(f'best_{model_name}.h5', save_best_only=True, monitor='val_auc', mode='max', verbose=1)
    ]
    
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=32,
        class_weight=class_weight_dict,
        callbacks=callbacks,
        verbose=1
    )
    
    # Evaluate on test set
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name}")
    print(f"{'='*60}")
    
    y_pred_prob = model.predict(X_test, verbose=0).ravel()
    y_pred = (y_pred_prob >= 0.5).astype(int)
    
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, digits=4))
    
    roc_auc = roc_auc_score(y_test, y_pred_prob)
    print(f"\nROC-AUC: {roc_auc:.4f}")
    
    return history, roc_auc

# ========== Train All Models ==========
results = {}

# Model : NO Attention CNN
print("\n" + "="*60)
print("MODEL 4: NO ATTENTION CNN")
print("="*60)
model4 = build_cnn_lstm(input_shape)
model4.summary()
history4, auc4 = train_and_evaluate(model4, "no_attention_cnn")
results['NO Attention CNN'] = auc4

# ========== Final Results ==========
print("\n" + "="*60)
print("FINAL RESULTS - ROC-AUC SCORES")
print("="*60)
for model_name, auc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{model_name:20s}: {auc:.4f}")

best_model = max(results, key=results.get)
print(f"\n🏆 BEST MODEL: {best_model} with AUC = {results[best_model]:.4f}")

Loading data...
Train shape: (1105, 128, 94, 1)
Val shape: (238, 128, 94, 1)
Test shape: (237, 128, 94, 1)
Class weights: {0: 1.0009057971014492, 1: 0.9990958408679927}

MODEL 4: NO ATTENTION CNN
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 128, 94, 1)]      0         
                                                                 
 conv2d (Conv2D)             (None, 128, 94, 64)       640       
                                                                 
 batch_normalization (BatchN  (None, 128, 94, 64)      256       
 ormalization)                                                   
                                                                 
 max_pooling2d (MaxPooling2D  (None, 64, 47, 64)       0         
 )                                                               
                                                               

### Testing data

### Validation data

In [5]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Load normalization stats (same as training)
stats = np.load("normalization_stats.npz")
mean, std = stats["mean"], stats["std"]

# Load test dataframe
test_df = pd.read_csv("test1.csv")   # <-- test_df explicitly used

# Data loader (same as training)
def load_data(df, mean, std, root_dir="."):
    X, y = [], []
    for _, row in df.iterrows():
        file_path = f"{root_dir}/{row['filename']}"
        mel = np.load(file_path).astype(np.float32)
        mel = mel[..., np.newaxis]
        mel = (mel - mean) / std
        X.append(mel)
        y.append(row['label'])
    return np.array(X), np.array(y)

# Load test data
X_test, y_test = load_data(test_df, mean, std)

# Load trained model
model = tf.keras.models.load_model("best_no_attention_cnn.h5")

# Evaluate model
results = model.evaluate(X_test, y_test, verbose=1, return_dict=True)

print("\nTest Results:")
for k, v in results.items():
    print(f"{k}: {v:.4f}")

# Predictions (binary classification – as in your files)
y_prob = model.predict(X_test)
y_pred = (y_prob > 0.5).astype(int).flatten()

# Metrics
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# ROC-AUC
roc_auc = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC Score: {roc_auc:.4f}")

8/8 [==============================] - 4s 386ms/step - loss: 0.0731 - accuracy: 0.9789 - precision: 0.9829 - recall: 0.9746 - auc: 0.9938

Test Results:
loss: 0.0731
accuracy: 0.9789
precision: 0.9829
recall: 0.9746
auc: 0.9938
8/8 [==============================] - 4s 376ms/step

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.98      0.98       119
           1       0.98      0.97      0.98       118

    accuracy                           0.98       237
   macro avg       0.98      0.98      0.98       237
weighted avg       0.98      0.98      0.98       237

Confusion Matrix:
[[117   2]
 [  3 115]]
ROC-AUC Score: 0.9970


In [6]:

import numpy as np
import tensorflow as tf
import librosa

# Load model
model = tf.keras.models.load_model("best_no_attention_cnn.h5")

# Load normalization stats
stats = np.load("normalization_stats.npz")
mean, std = stats["mean"], stats["std"]

# Load unseen audio
audio_path = "crying_unseen_1.wav"
y, sr = librosa.load(audio_path, sr=16000)

# Compute mel spectrogram (same params as training)
mel = librosa.feature.melspectrogram(
    y=y,
    sr=sr,
    n_mels=128,
    n_fft=2048,
    hop_length=512
)
mel = librosa.power_to_db(mel, ref=np.max)

# -------- FIX SHAPE HERE --------
TARGET_FRAMES = 94
if mel.shape[1] > TARGET_FRAMES:
    mel = mel[:, :TARGET_FRAMES]
else:
    mel = np.pad(mel, ((0, 0), (0, TARGET_FRAMES - mel.shape[1])), mode="constant")

# Normalize
mel = (mel - mean) / std

# Add dims
mel = mel[..., np.newaxis]        # (128, 94, 1)
mel = np.expand_dims(mel, axis=0) # (1, 128, 94, 1)

# Predict
prob = model.predict(mel)[0][0]
pred = 1 if prob > 0.5 else 0

print(f"Probability: {prob:.4f}")
print(f"Predicted class: {pred}")

C:\Users\Pooja\AppData\Local\Programs\Python\Python310\lib\site-packages\librosa\core\intervals.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


1/1 [==============================] - 2s 2s/step
Probability: 0.9918
Predicted class: 1


In [7]:

import numpy as np
import tensorflow as tf
import librosa

# Load model
model = tf.keras.models.load_model("best_attention_cnn.h5")

# Load normalization stats
stats = np.load("normalization_stats.npz")
mean, std = stats["mean"], stats["std"]

# Load unseen audio
audio_path = "env_unseen_bal.wav"
y, sr = librosa.load(audio_path, sr=16000)

# Compute mel spectrogram (same params as training)
mel = librosa.feature.melspectrogram(
    y=y,
    sr=sr,
    n_mels=128,
    n_fft=2048,
    hop_length=512
)
mel = librosa.power_to_db(mel, ref=np.max)

# -------- FIX SHAPE HERE --------
TARGET_FRAMES = 94
if mel.shape[1] > TARGET_FRAMES:
    mel = mel[:, :TARGET_FRAMES]
else:
    mel = np.pad(mel, ((0, 0), (0, TARGET_FRAMES - mel.shape[1])), mode="constant")

# Normalize
mel = (mel - mean) / std

# Add dims
mel = mel[..., np.newaxis]        # (128, 94, 1)
mel = np.expand_dims(mel, axis=0) # (1, 128, 94, 1)

# Predict
prob = model.predict(mel)[0][0]
pred = 1 if prob > 0.5 else 0

print(f"Probability: {prob:.4f}")
print(f"Predicted class: {pred}")


1/1 [==============================] - 2s 2s/step
Probability: 0.0002
Predicted class: 0


In [10]:

import numpy as np
import tensorflow as tf
import librosa

# Load model
model = tf.keras.models.load_model("best_no_attention_cnn.h5")

# Load normalization stats
stats = np.load("normalization_stats.npz")
mean, std = stats["mean"], stats["std"]

# Load unseen audio
audio_path = "train_crying.wav"
y, sr = librosa.load(audio_path, sr=16000)

# Compute mel spectrogram (same params as training)
mel = librosa.feature.melspectrogram(
    y=y,
    sr=sr,
    n_mels=128,
    n_fft=2048,
    hop_length=512
)
mel = librosa.power_to_db(mel, ref=np.max)

# -------- FIX SHAPE HERE --------
TARGET_FRAMES = 94
if mel.shape[1] > TARGET_FRAMES:
    mel = mel[:, :TARGET_FRAMES]
else:
    mel = np.pad(mel, ((0, 0), (0, TARGET_FRAMES - mel.shape[1])), mode="constant")

# Normalize
mel = (mel - mean) / std

# Add dims
mel = mel[..., np.newaxis]        # (128, 94, 1)
mel = np.expand_dims(mel, axis=0) # (1, 128, 94, 1)

# Predict
prob = model.predict(mel)[0][0]
pred = 1 if prob > 0.5 else 0

print(f"Probability: {prob:.4f}")
print(f"Predicted class: {pred}")

1/1 [==============================] - 2s 2s/step
Probability: 0.9994
Predicted class: 1


In [11]:

import numpy as np
import tensorflow as tf
import librosa

# Load model
model = tf.keras.models.load_model("best_no_attention_cnn.h5")

# Load normalization stats
stats = np.load("normalization_stats.npz")
mean, std = stats["mean"], stats["std"]

# Load unseen audio
audio_path = "train_env.wav"
y, sr = librosa.load(audio_path, sr=16000)

# Compute mel spectrogram (same params as training)
mel = librosa.feature.melspectrogram(
    y=y,
    sr=sr,
    n_mels=128,
    n_fft=2048,
    hop_length=512
)
mel = librosa.power_to_db(mel, ref=np.max)

# -------- FIX SHAPE HERE --------
TARGET_FRAMES = 94
if mel.shape[1] > TARGET_FRAMES:
    mel = mel[:, :TARGET_FRAMES]
else:
    mel = np.pad(mel, ((0, 0), (0, TARGET_FRAMES - mel.shape[1])), mode="constant")

# Normalize
mel = (mel - mean) / std

# Add dims
mel = mel[..., np.newaxis]        # (128, 94, 1)
mel = np.expand_dims(mel, axis=0) # (1, 128, 94, 1)

# Predict
prob = model.predict(mel)[0][0]
pred = 1 if prob > 0.5 else 0

print(f"Probability: {prob:.4f}")
print(f"Predicted class: {pred}")

1/1 [==============================] - 3s 3s/step
Probability: 0.0006
Predicted class: 0


In [ ]:

import numpy as np
import tensorflow as tf
import librosa

# Load model
model = tf.keras.models.load_model("best_no_attention_cnn.h5")

# Load normalization stats
stats = np.load("normalization_stats.npz")
mean, std = stats["mean"], stats["std"]

# Load unseen audio
audio_path = "train_crying.wav"
y, sr = librosa.load(audio_path, sr=16000)

# Compute mel spectrogram (same params as training)
mel = librosa.feature.melspectrogram(
    y=y,
    sr=sr,
    n_mels=128,
    n_fft=2048,
    hop_length=512
)
mel = librosa.power_to_db(mel, ref=np.max)

# -------- FIX SHAPE HERE --------
TARGET_FRAMES = 94
if mel.shape[1] > TARGET_FRAMES:
    mel = mel[:, :TARGET_FRAMES]
else:
    mel = np.pad(mel, ((0, 0), (0, TARGET_FRAMES - mel.shape[1])), mode="constant")

# Normalize
mel = (mel - mean) / std

# Add dims
mel = mel[..., np.newaxis]        # (128, 94, 1)
mel = np.expand_dims(mel, axis=0) # (1, 128, 94, 1)

# Predict
prob = model.predict(mel)[0][0]
pred = 1 if prob > 0.5 else 0

print(f"Probability: {prob:.4f}")
print(f"Predicted class: {pred}")

In [ ]:
# Independent test cell for VS Code Jupyter

# 1. Simple print output
print("VS Code Jupyter is working correctly!")

# 2. Basic calculation
a = 10
b = 5
print("Sum:", a + b)
print("Product:", a * b)

# 3. Generate a simple plot
import matplotlib.pyplot as plt

x = [1, 2, 3, 4, 5]
y = [i**2 for i in x]

plt.figure()
plt.plot(x, y)
plt.title("Test Plot")
plt.xlabel("X values")
plt.ylabel("Squared values")
plt.show()

# 4. Show current Python version
import sys
print("Python version:", sys.version)
